# finance / portfolio / utility

**Intent**
- Takes `finance_portfolio_utility_wealth`, a wealth multiple.
- Outputs `finance_portfolio_utility_value`, constant relative risk aversion utility.
- Ranks portfolios and decisions by what they are worth to you, not by their average.
- Acceptance holds the rmse on `validation` and `backtest` to 0.001.

**Data**
- Personal data is git-ignored in `data/`, with synthetic placeholders committed.
- `training` is ten wealth multiples with their utility, `validation` is five larger, and `backtest` is ten across the range.
- `backtest` is held back for acceptance.
- `golden` is multiples from half to ten, then a request with the input missing.

**Status**
- The risk aversion is recovered from synthetic utilities, and the real input is your own choices.

**Future**
- Add `query_postgres` to `workbench.data` and load the real series in place of the synthetic ones.
- Elicit risk aversion from certainty equivalents of 50/50 gambles, for you and your partner.
- Ask twice, a year apart, since a risk aversion that moves with the day is noise.
- Report a low, central and high risk aversion, not a point.
- Fix the limits against real data, then register and review before any release.

## Setup ([6.7.1](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#671-setup))

In [ ]:
import json
from os.path import join

from workbench.config import NOTEBOOK_DIR
from workbench.data import ensure_data, golden_data
from workbench.env import init_env
from workbench.project import load_signature, provenance, record_project, scratch_project_dir, signature_names

env = init_env()

WORKSPACE = "finance/portfolio/utility"
MODEL_NAME = "finance_portfolio_utility"
TARGET = {"kind": "local"}
signature = load_signature(WORKSPACE)
INPUTS, OUTPUTS = signature_names(signature)
TRAINING = "portfolio_utility_training_2026-11-05_2026-11-05"
VALIDATION = "portfolio_utility_validation_2026-11-05_2026-11-05"
BACKTEST = "portfolio_utility_backtest_2026-11-05_2026-11-05"
GOLDEN = "portfolio_utility_golden"

## Capture Data ([6.7.2](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#672-capture-data))

In [ ]:
from math import log


def utility(wealth, gamma=3.0):
    return log(wealth) if gamma == 1.0 else wealth ** (1.0 - gamma) / (1.0 - gamma)


def labelled(wealths):
    return {INPUTS[0]: wealths, OUTPUTS[0]: [round(utility(wealth), 6) for wealth in wealths]}


training_pl, training_path = ensure_data(WORKSPACE, "training", TRAINING, lambda: labelled([0.5, 0.75, 1.0, 1.25, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0]))
validation_pl, validation_path = ensure_data(WORKSPACE, "validation", VALIDATION, lambda: labelled([6.0, 7.0, 8.0, 9.0, 10.0]))
backtest_pl, backtest_path = ensure_data(WORKSPACE, "backtest", BACKTEST, lambda: labelled([0.6, 0.9, 1.1, 1.7, 2.2, 3.3, 3.7, 4.5, 4.8, 12.0]))
print("training", training_pl.height, "validation", validation_pl.height, "backtest", backtest_pl.height)

## Fit ([6.7.4](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#674-fit))

In [ ]:
from scipy.optimize import curve_fit

fitted_gamma = curve_fit(lambda wealth, gamma: wealth ** (1.0 - gamma) / (1.0 - gamma), training_pl[INPUTS[0]].to_numpy(), training_pl[OUTPUTS[0]].to_numpy(), p0=[2.0], bounds=(1.5, 10.0))[0][0]
PARAMS = {"gamma": round(float(fitted_gamma), 2)}
print("fitted", PARAMS)
assert PARAMS == {"gamma": 3.0}, "the fit did not recover the risk aversion"

## Evaluate ([6.7.5](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#675-evaluate))

In [ ]:
from workbench.model import evaluate_acceptance, measure
from workbench.replay import model_predict

MODEL_TEMPLATE = """from math import log

import pandas as pd
from mlflow.models import set_model
from mlflow.pyfunc import PythonModel

INPUT = {inputs[0]!r}
OUTPUT = {outputs[0]!r}
GAMMA = {gamma!r}


def utility(wealth):
    return log(wealth) if GAMMA == 1.0 else wealth ** (1.0 - GAMMA) / (1.0 - GAMMA)


class Utility(PythonModel):
    def predict(self, context, model_input: pd.DataFrame, params=None):
        return pd.DataFrame({{OUTPUT: [utility(wealth) for wealth in model_input[INPUT]]}})


set_model(Utility())
"""

model_path = join(NOTEBOOK_DIR, WORKSPACE, "model.py")
with open(model_path, "w") as model_file:
    model_file.write(MODEL_TEMPLATE.format(**PARAMS, inputs=INPUTS, outputs=OUTPUTS))

predict = model_predict(model_path, "pyfunc", signature)

In [ ]:
LIMIT = 0.001
SPLITS = (("validation", validation_pl), ("backtest", backtest_pl))


def acceptance_checks():
    answers = {split: predict({INPUTS[0]: frame[INPUTS[0]].to_list()}) for split, frame in SPLITS}
    return [
        {"split": split, "output": output, "name": "rmse", "value": measure(frame[output].to_list(), answers[split][output])["rmse"], "limit": LIMIT, "direction": "max"}
        for split, frame in SPLITS
        for output in OUTPUTS
    ]


evaluation = evaluate_acceptance(acceptance_checks, {"validation": [validation_path], "backtest": [backtest_path]})
for entry in evaluation["checks"]:
    print(entry["split"], entry["output"], entry["name"], "measured", entry["value"], "passed", entry["passed"])
print("acceptance", "passed" if evaluation["passed"] else "failed, the shipped model is outside the limit")

## Package ([6.7.6](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#676-package))

In [ ]:
def golden_samples():
    xs = [0.5, 1.0, 1.5, 2.0, 5.0, 10.0]
    answers = predict({INPUTS[0]: xs})
    return {INPUTS[0]: [*xs, None], **{output: [*answers[output], None] for output in OUTPUTS}, "expect": ["ok"] * len(xs) + ["error"]}

In [ ]:
from workbench.model import package_project
from workbench.replay import require_replay

golden_pl, golden_path = golden_data(WORKSPACE, GOLDEN, golden_samples)

project_dir = scratch_project_dir(WORKSPACE)
manifest = package_project(
    project_dir,
    MODEL_NAME,
    "pyfunc",
    signature,
    golden_pl.iter_rows(named=True),
    provenance(
        "manual",
        "synthetic constant relative risk aversion utilities, training split",
        [training_path, golden_path],
        "least squares risk aversion",
        implementation="mlflow.pyfunc model.py, requires mlflow and pandas",
    ),
    model_path=model_path,
    evaluation=evaluation,
    sequence="across losses, today's wealth and large gains",
)
print(json.dumps(manifest, indent=2))

require_replay(project_dir)
metadata = record_project(WORKSPACE, project_dir, TARGET)
print("replayed the golden sequences exactly through the shipped runtime")